In [0]:
-- Create Silver streaming table for cleaned and validated order items
CREATE OR REFRESH STREAMING TABLE first_data_engineering_project.silver.silver_order_items

-- Describe the purpose of the Silver table
COMMENT "Cleaned and validated order items table"

-- Define Silver table properties
TBLPROPERTIES (
    "quality" = "silver",
    "pipelines.reset.allowed" = false
)

-- Define data quality expectations
(
    -- Order item ID must exist and be a positive value
    CONSTRAINT valid_order_item_id
        EXPECT (order_item_id IS NOT NULL AND order_item_id > 0)
        ON VIOLATION DROP ROW,

    -- Order ID must exist and be a positive value
    CONSTRAINT valid_order_id
        EXPECT (order_id IS NOT NULL AND order_id > 0)
        ON VIOLATION DROP ROW,

    -- Product ID must exist and be a positive value
    CONSTRAINT valid_product_id
        EXPECT (product_id IS NOT NULL AND product_id > 0)
        ON VIOLATION DROP ROW,

    -- Quantity must exist and be greater than zero
    CONSTRAINT valid_qty
        EXPECT (qty IS NOT NULL AND qty > 0)
        ON VIOLATION DROP ROW,

    -- Price must exist and cannot be negative
    CONSTRAINT valid_price
        EXPECT (price IS NOT NULL AND price >= 0)
        ON VIOLATION DROP ROW
)

AS

-- Deduplicate order items and keep the most recently ingested record
WITH deduplicated_order_items AS (

    SELECT
        *,

        -- Assign row number 1 to the most recently ingested
        -- record for each order_item_id
        ROW_NUMBER() OVER (
            PARTITION BY order_item_id
            ORDER BY ingestion_timestamp DESC
        ) AS row_num

    -- Read order items from the Bronze streaming table
    FROM STREAM first_data_engineering_project.bronze.bronze_order_items
)

-- Select the cleaned Silver columns
SELECT
    order_item_id,
    order_id,
    product_id,
    qty,
    price,
    ingestion_timestamp,
    source_file,
    file_modified_time

-- Read the deduplicated order items
FROM deduplicated_order_items

-- Keep only the most recently ingested record for each order_item_id
WHERE row_num = 1;